In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

from matplotlib import pyplot as plt
from matplotlib_venn import venn3

import statsmodels.api as sm

import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Read data

In [2]:
df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/output_eadms_sintetic_07_05_2026.parquet')

# Run MV and logistic regression

In [3]:
#df.columns.to_list()

In [8]:
def lr_mv(data, replicates=range(32), lags=[1, 2, 3]):

    results = []

    for rep in replicates:

        print(f"\n===== Processing replicate {rep} =====")

        # -----------------------------
        # Dynamic column names
        # -----------------------------
        lst_models = [
            f'C2_alarms_{rep}',
            f'sinal_evi_replicate_{rep}',
            f'EWS_ISF_replicate_{rep}',
            f'EWS_LOF_replicate_{rep}',
            f'EWS_OCSVM_replicate_{rep}',
            f'EWS_COPOD_replicate_{rep}',
            f'EWS_Rt_replicate_{rep}'
        ]

        col_surge = f'mem_surge_01_replicate_{rep}'
        col_surge_consec = f'mem_surge_01_replicate_{rep}_correct_with_consec'

        # -----------------------------
        # Create lagged column names
        # -----------------------------
        lagged_cols = []

        for var in lst_models:
            for lag in lags:
                lagged_cols.append(f"{var}_lag_{lag}")

        all_features = lst_models + lagged_cols

        # -----------------------------
        # Ensure binary integer type
        # -----------------------------
        data[lst_models] = data[lst_models].astype(int)

        # -----------------------------
        # Majority voting
        # -----------------------------
        threshold = round(len(lst_models) / 2, 0)

        lst_cities = []

        for code in data.co_ibge.unique():

            print(f"Processing MV for {code}...")

            set_muni = (
                data[data.co_ibge == code]
                .sort_values(['year_week'])
                .copy()
            )

            set_muni[f'hard_voting_ivas_rep_{rep}'] = (
                set_muni[lst_models].sum(axis=1) >= threshold
            ).astype(int)

            lst_cities.append(set_muni)

        data_rep = pd.concat(lst_cities)

        # -----------------------------
        # Logistic regression
        # -----------------------------
        lst = []

        for code in data_rep.co_ibge.unique():

            print(f"Processing LR for {code}...")

            set_muni = (
                data_rep[data_rep.co_ibge == code]
                .sort_values(['year_week'])
                .copy()
            )

            # Create lagged features
            for var in lst_models:
                for lag in lags:

                    set_muni[f"{var}_lag_{lag}"] = (
                        set_muni[var]
                        .shift(lag)
                        .fillna(0)
                        .astype(int)
                    )

            # Train only if enough positivesif set_muni["warning_final_mem_surge_01"].sum() > 1:
            if set_muni[col_surge].sum() > 1 and set_muni[col_surge_consec].nunique() > 1:

                X = set_muni[all_features].fillna(0)
                y = set_muni[col_surge_consec]

                try:

                    X_train, X_test, y_train, y_test = train_test_split(
                        X,
                        y,
                        test_size=0.2,
                        random_state=500,
                        stratify=y
                    )

                    clf = LogisticRegression(
                        class_weight="balanced",
                        max_iter=500
                    )

                    clf.fit(X_train, y_train)

                    set_muni[f"risk_probs_rep_{rep}"] = (
                        clf.predict_proba(X)[:, 1]
                    )

                    set_muni[f"sinal_ens_ivas_rep_{rep}"] = (
                        set_muni[f"risk_probs_rep_{rep}"] > 0.5
                    ).astype(int)

                except Exception as e:

                    print(f"Error for {code}, rep {rep}: {e}")

                    set_muni[f"sinal_ens_ivas_rep_{rep}"] = (
                        set_muni[f'hard_voting_ivas_rep_{rep}']
                    )

            else:

                set_muni[f"sinal_ens_ivas_rep_{rep}"] = (
                    set_muni[f'hard_voting_ivas_rep_{rep}']
                )

            lst.append(set_muni)

        rep_result = pd.concat(lst)

        # Keep only new columns to avoid duplication
        keep_cols = [
            'co_ibge',
            'year_week',
            f'hard_voting_ivas_rep_{rep}',
            f'risk_probs_rep_{rep}',
            f'sinal_ens_ivas_rep_{rep}'
        ]

        keep_cols = [c for c in keep_cols if c in rep_result.columns]

        results.append(rep_result[keep_cols])

    # ---------------------------------
    # Merge all replicate results
    # ---------------------------------
    final_res = data.copy()

    for df_rep in results:
        final_res = final_res.merge(
            df_rep,
            on=['co_ibge', 'year_week'],
            how='left'
        )

    return final_res

In [9]:
#final_df = lr_mv(df, replicates=[0,1])
final_df = lr_mv(df)


===== Processing replicate 0 =====
Processing MV for 110001...
Processing MV for 110002...
Processing MV for 110005...
Processing MV for 110007...
Processing MV for 110008...
Processing MV for 110009...
Processing MV for 110010...
Processing MV for 110011...
Processing MV for 110013...
Processing MV for 110014...
Processing MV for 110015...
Processing MV for 110018...
Processing MV for 110020...
Processing MV for 110025...
Processing MV for 110026...
Processing MV for 110028...
Processing MV for 110029...
Processing MV for 110030...
Processing MV for 110032...
Processing MV for 110033...
Processing MV for 110034...
Processing MV for 110037...
Processing MV for 110040...
Processing MV for 110045...
Processing MV for 110050...
Processing MV for 110060...
Processing MV for 110080...
Processing MV for 110090...
Processing MV for 110092...
Processing MV for 110094...
Processing MV for 110100...
Processing MV for 110110...
Processing MV for 110120...
Processing MV for 110130...
Processing M

KeyboardInterrupt: 

In [ ]:
dta =  final_df.isnull().sum().reset_index()

In [ ]:
dta[dta[0] >= 1]

In [ ]:
from pathlib import Path
from datetime import datetime

out_dir = Path("/opt/storage/shared/aesop/aesop_shared/ensamble_modelling")

fname = f"LR_MV_sintetic_{datetime.now():%d_%m_%Y}.parquet"

final_df.to_parquet(out_dir / fname)